[◀ Notebook 05: Compound Statements](05_compound_statements.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 07: Functions & Scoping ▶](07_functions_and_scoping.ipynb)

# Notebook 06: Built-in Data Structures

This notebook explores Python's primary built-in data structures—**Lists**, **Tuples**, **Dictionaries**, and **Sets**—with an emphasis on their under-the-hood CPython implementation details and advanced usage patterns.

Reference: [Built-in Types](https://docs.python.org/3/library/stdtypes.html)


## 1. Lists: Mutable Dynamic Arrays

### CPython Implementation
In CPython, a `list` is implemented as a **contiguous array of pointers** to other Python objects, rather than an array of the object values themselves. This design has important performance implications:
- **$O(1)$ Time Complexity** for indexing (`list[i]`) and length queries (`len(list)`).
- **Amortized $O(1)$ Time Complexity** for append operations (`list.append(item)`). CPython overallocates memory slot capacity dynamically to minimize the overhead of memory reallocations. The growth pattern is roughly: `size_with_over_allocation = size + (size >> 3) + (size < 9 ? 3 : 6)`.
- **$O(n)$ Time Complexity** for inserts and deletions (`list.insert()`, `list.pop(0)`), since all subsequent pointers in the array must be shifted in memory.

### Advanced Slicing
Slicing in Python follows the syntax `sequence[start:stop:step]`, which evaluates to creating a built-in `slice(start, stop, step)` object and passing it to the object's `__getitem__` method.


In [ ]:
import sys

# 1. Demonstrate list over-allocation (capacity growth)
lst = []
prev_size = sys.getsizeof(lst)
print(f"Initial empty list memory size: {prev_size} bytes")
for i in range(15):
    lst.append(i)
    curr_size = sys.getsizeof(lst)
    if curr_size != prev_size:
        print(f"Size changed to: {curr_size} bytes (Elements count: {len(lst)})")
        prev_size = curr_size

# 2. Advanced Slicing (list[start:stop:step] with negative step/slices)
items = [0, 1, 2, 3, 4, 5]
print("\nAdvanced Slicing:")
print("Reverse sequence using step -1 (items[::-1]):", items[::-1])
print("Slice step -2 (items[4:1:-2]):", items[4:1:-2])


## 2. Tuples: Immutable Sequences

### Memory and Recycling Optimizations
Tuples are immutable sequences. Once created, their size and element bindings cannot change. This immutability allows CPython to apply memory optimizations:
- **Recycling Small Tuples**: To speed up allocation, CPython maintains a free list pool of deallocated tuple objects of length up to 20. Re-creating a tuple of these sizes often recycles cached memory rather than allocating fresh blocks.
- **Hashability**: A tuple is hashable **only if** all of its contained items are also hashable. This allows tuples to be used as dictionary keys or set elements, unlike lists.


In [ ]:
# 1. Hashability of Tuples
tup_hashable = (1, 2, "three")
print(f"Hash of {tup_hashable}: {hash(tup_hashable)}")

try:
    tup_unhashable = (1, 2, [3, 4])  # Contains a mutable list
    hash(tup_unhashable)
except TypeError as e:
    print(f"TypeError raised as expected: {e}")

# 2. Unpacking and target lists
a, *b, c = (1, 2, 3, 4, 5)
print(f"Unpacking: a={a}, b={b}, c={c}")


## 3. Dictionaries: Ordered Mappings

### Key Hashability and PEP 468 Layout
Dictionaries (`dict`) map hashable keys to arbitrary values. Keys must implement `__hash__` and `__eq__` methods.
Starting with Python 3.6 (and codified in Python 3.7 via PEP 468), CPython uses a compact hash table layout:
- **Compact Dicts**: Instead of a sparse table of 24-byte entries containing hashes, keys, and values, CPython stores entries in a contiguous **dense array** and keeps a separate, smaller **indices array** of integers (which act as hash buckets). This reduces memory consumption by 30% to 95% and naturally preserves key insertion order.

### Dictionary Views
The methods `.keys()`, `.values()`, and `.items()` return dynamic **dictionary view objects**.
- Keys and items views behave like **sets** (they support set operations like union, intersection, and difference) because keys are guaranteed unique and hashable.

### CPython Internal Mechanics of Dictionary Views
In CPython, dictionary views are lightweight objects (`PyDictKeys_Type`, `PyDictValues_Type`, `PyDictItems_Type`) that maintain a reference to the underlying dictionary. They do not duplicate the dictionary's keys or values in memory. Instead, they provide a dynamic window into the dictionary. When you perform set operations on `dict_keys` or `dict_items` views, CPython internally executes set operations by leveraging the hash table structure of the keys. Specifically, `dict_keys` implements the set operators. For example, for the difference operator (`-`), CPython iterates over the view and checks membership in the other operand using its internal hash lookup, returning a standard `set` object.

### Rebuilding Dictionaries from View Differences Using Dict Comprehensions
Since set operations on dictionary views return standard Python `set` objects, they can be utilized directly within dict comprehensions to rebuild dictionaries with filtered keys. This is highly efficient because it avoids manual membership checking loops in Python, leveraging C-level set operations and hash lookups first.

```python
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'b': 20, 'c': 3, 'd': 4}

# 1. Rebuild dict with keys only in d1 (Difference: d1.keys() - d2.keys())
removed_dict = {k: d1[k] for k in d1.keys() - d2.keys()}
# Result: {'a': 1}

# 2. Rebuild dict with keys common to both (Intersection: d1.keys() & d2.keys())
common_dict = {k: d1[k] for k in d1.keys() & d2.keys()}
# Result: {'b': 2, 'c': 3}

# 3. Rebuild dict with combined keys (Union: d1.keys() | d2.keys())
# Prioritize values from d2 in case of overlaps
merged_dict = {k: (d2[k] if k in d2 else d1[k]) for k in d1.keys() | d2.keys()}
# Result: {'a': 1, 'b': 20, 'c': 3, 'd': 4}
```


In [ ]:
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'b': 20, 'c': 3, 'd': 4}

# Get keys view objects
keys1 = d1.keys()
keys2 = d2.keys()

# Perform set operations directly on views!
print("Common keys (intersection):", keys1 & keys2)
print("Keys only in d1 (difference):", keys1 - keys2)
print("All keys combined (union):", keys1 | keys2)

# Dynamic updates (dictionary views update automatically when dict changes)
print("\nKeys view before dict modification:", keys1)
d1['new_key'] = 99
print("Keys view after dict modification:", keys1)


## 4. Sets: Unordered Unique Collections

Sets are mutable collections of unique, hashable objects. Under the hood, a set is implemented similarly to a dictionary, but without storing values (it only keeps keys). Set operations utilize high-speed hash-table operations, making membership testing (`in`) highly efficient ($O(1)$ time complexity on average).


In [ ]:
set_a = {1, 2, 3, 4}
set_b = {3, 4, 5, 6}

# Set Algebra operators vs. methods
print("Union (|):", set_a | set_b)                  # Equivalent to set_a.union(set_b)
print("Intersection (&):", set_a & set_b)            # Equivalent to set_a.intersection(set_b)
print("Difference (-):", set_a - set_b)              # Equivalent to set_a.difference(set_b)
print("Symmetric Difference (^):", set_a ^ set_b)    # Equivalent to set_a.symmetric_difference(set_b)


## 5. Sequence Looping & Iteration Techniques

When iterating over sequences and collections, Python provides built-in utilities that abstract away manual index management and lock-step parallel iteration. These return lightweight iterator objects that evaluate lazily on demand ($O(1)$ space complexity):

-   **`enumerate(iterable, start=0)`:** Returns an enumerate iterator yielding tuples of `(index, value)`. This is the preferred Pythonic way to track loop indices without resorting to manual counters.
-   **`zip(*iterables, strict=False)`:** Aggregates elements from multiple iterables in parallel, yielding tuples. If `strict=True` (introduced in Python 3.10), it raises a `ValueError` if the iterables are not of equal length. By default (`strict=False`), it stops silently when the shortest iterable is exhausted.
-   **`reversed(seq)`:** Returns a reverse iterator. The input must be a sequence (supporting `__len__` and `__getitem__`) or implement `__reversed__`. It does *not* copy the data, yielding items from the end to the start.
-   **`sorted(iterable, key=None, reverse=False)`:** Returns a **new sorted list** containing all items. Unlike `list.sort()`, which mutates in-place, `sorted()` is safe for immutable collections and returns a brand-new list using CPython's optimized **Timsort** algorithm ($O(n \log n)$ time, $O(n)$ space).

In [ ]:
# 1. Demonstrate enumerate()
colors = ["red", "green", "blue"]
print("Enumerate output:")
for index, color in enumerate(colors, start=1):
    print(f"  Color #{index}: {color}")

# 2. Demonstrate zip() with strict=True check
names = ["Alice", "Bob"]
ages = [25, 30, 35]  # Mismatched length

print("\nZip (default, stops at shortest):")
for name, age in zip(names, ages):
    print(f"  {name} is {age} years old")

try:
    print("\nZip (strict=True, checks length):")
    for name, age in zip(names, ages, strict=True):
        print(f"  {name} is {age} years old")
except ValueError as e:
    print(f"  Caught expected ValueError: {e}")

# 3. Demonstrate reversed() vs sorted()
numbers = [5, 2, 9, 1]
rev_iter = reversed(numbers)
sorted_list = sorted(numbers)

print(f"\nOriginal list: {numbers}")
print(f"reversed() type is iterator: {rev_iter} -> List conversion: {list(rev_iter)}")
print(f"sorted() type is list: {sorted_list}")
print(f"Original list remains unchanged: {numbers}")

## 6. List Comprehensions and Scope Isolation

List comprehensions (PEP 202) provide a concise way to create lists.

**Under-the-hood namespace isolation:** In Python 2, list comprehensions leaked loop control variables (e.g., `x`) to the surrounding scope. In Python 3, list comprehensions are executed inside a **separate, temporary nested scope** (similar to a function call). Hence, loop variables are fully isolated and do not leak.

### Nested List Comprehensions and Evaluation Order
Nested list comprehensions can take two primary forms, which differ significantly in evaluation order and control flow.

#### Form 1: Multiple `for` Clauses (Flat Output)
When a list comprehension contains multiple `for` clauses, it is evaluated like nested loops. The leftmost `for` clause is the outermost loop, and the rightmost is the innermost.
```python
# List comprehension:
flat = [x for row in matrix for x in row]

# Equivalent nested loops:
flat = []
for row in matrix:      # Outermost
    for x in row:       # Innermost
        flat.append(x)
```
*CPython Execution Flow:* CPython compiles this into a single code object. It creates a single stack frame where nested `FOR_ITER` bytecode instructions execute the loops sequentially.

#### Form 2: Nested Comprehensions (Nested Output)
When the expression of a list comprehension is itself another list comprehension, they are nested. The outer comprehension acts as the outer loop, and for each iteration, the inner comprehension is executed to produce a nested list.
```python
# Nested list comprehension:
nested = [[row[i] for row in matrix] for i in range(3)]

# Equivalent nested loops:
nested = []
for i in range(3):                     # Outer loop
    inner_list = []
    for row in matrix:                 # Inner loop
        inner_list.append(row[i])
    nested.append(inner_list)
```
*CPython Execution Flow:* CPython compiles the outer comprehension into a code object, and the inner comprehension into a *separate* nested code object. During execution, the outer comprehension's loop calls the inner comprehension's function object on every iteration, creating and tearing down stack frames for the inner list construction.

#### Application: Matrix Transposition
To transpose a matrix (swap its rows and columns), we iterate over the column indices in the outer loop, and for each column index, we extract the elements from each row in the inner loop:
```python
matrix = [
    [1, 2, 3],
    [4, 5, 6]
]
transposed = [[row[i] for row in matrix] for i in range(len(matrix[0]))]
# Output: [[1, 4], [2, 5], [3, 6]]
```
*Evaluation Steps:*
1. `range(len(matrix[0]))` evaluates to `range(3)`.
2. `i` is bound to `0`. The inner comprehension `[row[0] for row in matrix]` runs, iterating through `matrix` rows:
   - `row` is `[1, 2, 3]`, yields `row[0] -> 1`
   - `row` is `[4, 5, 6]`, yields `row[0] -> 4`
   - Returns `[1, 4]`.
3. `i` is bound to `1`. Inner comprehension yields `[2, 5]`.
4. `i` is bound to `2`. Inner comprehension yields `[3, 6]`.
5. Outer comprehension returns `[[1, 4], [2, 5], [3, 6]]`.

#### Alternative: Zip Unpacking (`zip(*matrix)`)
An alternative, highly Pythonic way to transpose a matrix is using `zip` with argument unpacking (`*` operator):
```python
matrix = [[1, 2, 3], [4, 5, 6]]
transposed = [list(row) for row in zip(*matrix)]
# Output: [[1, 4], [2, 5], [3, 6]]
```
This unpacks the outer list into separate rows, which are passed as positional arguments to `zip`. The `zip` function aggregates elements column-by-column, effectively transposing the matrix in optimized C code.

In [ ]:
x = "global scope string"

# Comprehension variable 'x' is isolated in Python 3
squared = [x ** 2 for x in range(5)]

print("Squared:", squared)
print("x after comprehension:", repr(x))  # Remains unchanged!

# Form 1: Multiple for clauses flat output list comprehension
matrix = [[1, 2], [3, 4]]
flat_list = [val for row in matrix for val in row]
print("\nFlat list comprehension output:", flat_list)

# Form 2: Nested output list comprehension (Matrix Transposition)
transposed = [[row[col] for row in matrix] for col in range(len(matrix[0]))]
print("Transposed matrix via nested comprehension:", transposed)

# Alternative: Transposition using zip unpacking
zip_transposed = [list(row) for row in zip(*matrix)]
print("Transposed matrix via zip unpacking:", zip_transposed)


## 7. Coding Challenges

Complete the following exercises to test your understanding.

### Challenge 1: Matrix Transposition using Nested List Comprehensions

Implement the function `transpose_matrix(matrix)` which takes a 2D list (a grid of lists) and returns its transpose (swapping rows and columns) using a single nested list comprehension.
Handling empty matrices `[]` should return `[]` safely.


In [ ]:
def transpose_matrix(matrix: list[list[any]]) -> list[list[any]]:
    """
    Transposes a 2D matrix (swapping rows and columns) using a single nested list comprehension.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    # Test case 1: Square matrix
    m1 = [[1, 2], [3, 4]]
    assert transpose_matrix(m1) == [[1, 3], [2, 4]], "Failed on square matrix"

    # Test case 2: Rectangular matrix
    m2 = [[1, 2, 3], [4, 5, 6]]
    assert transpose_matrix(m2) == [[1, 4], [2, 5], [3, 6]], "Failed on rectangular matrix"

    # Test case 3: Single element matrix
    m3 = [[42]]
    assert transpose_matrix(m3) == [[42]], "Failed on single element"

    # Test case 4: Empty matrix
    assert transpose_matrix([]) == [], "Failed on empty matrix"
    
    print("🎉 Challenge 1 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


### Challenge 2: Dictionary Diffing using Set Operations on Views

Implement `dict_diff(dict_a, dict_b)` which compares two dictionaries and returns a dictionary with three keys:
- `'added'`: a sub-dictionary of key-value pairs present in `dict_b` but not in `dict_a`.
- `'removed'`: a sub-dictionary of key-value pairs present in `dict_a` but not in `dict_b`.
- `'modified'`: a sub-dictionary where keys exist in both dictionaries, but have different values. The values in this sub-dictionary should be tuples of `(old_value, new_value)`.

**Constraint**: You MUST perform key and item comparisons using **set operations** on the dictionary's view objects (e.g., `keys()` or `items()`).


In [ ]:
def dict_diff(dict_a: dict, dict_b: dict) -> dict:
    """
    Compares dict_a and dict_b and returns the difference dictionary.
    Uses set operations on dict views.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    d1 = {'a': 1, 'b': 2, 'c': 3}
    d2 = {'b': 20, 'c': 3, 'd': 4}
    
    diff = dict_diff(d1, d2)
    
    assert diff['added'] == {'d': 4}, f"Expected added {{'d': 4}}, got {diff.get('added')}"
    assert diff['removed'] == {'a': 1}, f"Expected removed {{'a': 1}}, got {diff.get('removed')}"
    assert diff['modified'] == {'b': (2, 20)}, f"Expected modified {{'b': (2, 20)}}, got {diff.get('modified')}"
    
    # Test case with no differences
    assert dict_diff(d1, d1) == {'added': {}, 'removed': {}, 'modified': {}}
    
    print("🎉 Challenge 2 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


[◀ Notebook 05: Compound Statements](05_compound_statements.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 07: Functions & Scoping ▶](07_functions_and_scoping.ipynb)